# Tuần 3 — Neo kết quả trên Google Colab

**Trước khi chạy:** Runtime → Change runtime type → **GPU** (T4 hoặc tốt hơn).

| Ô | Việc | Thời gian |
|---|---|---|
| 1–2 | Kiểm tra GPU, mount Drive | 1 phút |
| 3 | Cài thư viện | 3–5 phút |
| 4 | Tải LibriSpeech | 15–25 phút (lần đầu) |
| 5 | Giải nén | 5–8 phút |
| 6 | Upload code | 1 phút |
| 7 | Dựng manifest | 3–5 phút |
| 8 | **Neo kết quả** | **40–70 phút** |
| 9 | Đo thời gian | 10 phút |

Tổng lần đầu: **~1.5–2 giờ**. Các lần sau bỏ qua ô 4 → còn ~1.5 giờ.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi
import torch
print("CUDA:", torch.cuda.is_available())
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHONG CO")
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

## 2. Mount Google Drive

Drive dùng để **lưu file .tar.gz** (tải một lần, dùng mãi) và **lưu kết quả**.

⚠️ **Không** giải nén dataset vào Drive — I/O của Drive rất chậm, đọc 28.500 file flac sẽ mất hàng giờ. Luôn giải nén vào `/content`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/alasr'
os.makedirs(f'{DRIVE}/tarballs', exist_ok=True)
os.makedirs(f'{DRIVE}/runs', exist_ok=True)
print('Drive:', DRIVE)

## 3. Cài thư viện

In [ ]:
!pip install -q transformers==4.36.2 datasets==2.16.1 accelerate==0.25.0 \
                 soundfile==0.12.1 jiwer==3.0.3 librosa==0.10.1
print("Xong. Neu Colab bao restart runtime -> restart roi chay tiep tu o 4.")

## 4. Tải LibriSpeech

Lần đầu tải về Drive. Các lần sau ô này tự bỏ qua.

In [ ]:
import os

FILES = {
    'train-clean-100.tar.gz': 'https://www.openslr.org/resources/12/train-clean-100.tar.gz',
    'test-clean.tar.gz':      'https://www.openslr.org/resources/12/test-clean.tar.gz',
    'test-other.tar.gz':      'https://www.openslr.org/resources/12/test-other.tar.gz',
}

for name, url in FILES.items():
    dst = f'{DRIVE}/tarballs/{name}'
    if os.path.exists(dst):
        print(f'DA CO  {name}  ({os.path.getsize(dst)/1e9:.2f} GB)')
    else:
        print(f'TAI    {name} ...')
        !wget -q --show-progress -O "$dst" "$url"

## 5. Giải nén vào /content (nhanh, không dùng Drive)

In [ ]:
!mkdir -p /content/data
for name in FILES:
    print('Giai nen', name, '...')
    !tar -xzf "{DRIVE}/tarballs/{name}" -C /content/data

!du -sh /content/data/LibriSpeech/*
!df -h /content | tail -1

## 6. Đưa code lên Colab

**Cách A** — nén thư mục `alasr/` trên máy thành `alasr.zip` rồi upload.

In [ ]:
from google.colab import files
up = files.upload()          # chon alasr.zip
!unzip -q -o alasr.zip -d /content/
%cd /content/alasr
!ls -R | head -30

**Cách B** — nếu đã đẩy code lên GitHub (khuyến nghị, tiện hơn nhiều)

In [ ]:
# %cd /content
# !git clone https://github.com/<username>/alasr.git
# %cd /content/alasr

## 7. Dựng manifest

In [ ]:
%cd /content/alasr
for split in ['train-clean-100', 'test-clean', 'test-other']:
    !python scripts/prepare_manifest.py \
        --librispeech-root /content/data/LibriSpeech \
        --split {split} --out-dir data/librispeech

**Kiểm tra bắt buộc** cho `train-clean-100`:

| Chỉ số | Giá trị đúng |
|---|---|
| Số utterance | **28.539** |
| Tổng thời lượng | **~100.6 giờ** |
| Số speaker | **251** |

Lệch nhiều → giải nén thiếu hoặc sai đường dẫn.

## 8. ⭐ NEO KẾT QUẢ

**Điều chỉnh batch size theo GPU** trước khi chạy — LibriSpeech có câu trung bình 12.7 giây, khá dài:

| GPU | VRAM | `per_device_train_batch_size` | `gradient_accumulation_steps` |
|---|---|---|---|
| T4 | 16 GB | **4** | **4** |
| L4 / V100 | 22–32 GB | 8 | 2 |
| A100 | 40–80 GB | 16 | 1 |

Sửa trong `models/w2v2_ctc.py`, phần `TrainingArguments`.

In [ ]:
# Chay thu 300 buoc truoc (~8 phut) de chac chan khong loi
!python scripts/anchor_run.py \
    --train-manifest    data/librispeech/train-clean-100.manifest.jsonl \
    --train-transcripts data/librispeech/train-clean-100.transcripts.jsonl \
    --test-manifest     data/librispeech/test-clean.manifest.jsonl \
    --test-transcripts  data/librispeech/test-clean.transcripts.jsonl \
    --hours 1.0 --seed 0 --max-steps 300 --out-dir runs/smoke

Chạy trơn → chạy thật (bỏ `--max-steps`, để script tự tính 2000 bước):

In [ ]:
!python scripts/anchor_run.py \
    --train-manifest    data/librispeech/train-clean-100.manifest.jsonl \
    --train-transcripts data/librispeech/train-clean-100.transcripts.jsonl \
    --test-manifest     data/librispeech/test-clean.manifest.jsonl \
    --test-transcripts  data/librispeech/test-clean.transcripts.jsonl \
    --hours 1.0 --seed 0 --out-dir runs/anchor

!cp -r runs/anchor {DRIVE}/runs/     # luu ngay vao Drive

**Đối chiếu** — wav2vec 2.0 Table 9, BASE, 1 giờ, không LM: **24.5 / 29.7**

| WER | Kết luận |
|---|---|
| 20–30% | ✅ Đạt — sang ô 9 |
| 30–40% | ⚠️ Tăng `--max-steps` lên 4000 |
| > 40% | ❌ Pipeline có lỗi — xem bảng chẩn đoán trong `anchor_run.py` |

## 9. Đo thời gian và tính lại ngân sách GPU

In [ ]:
!python scripts/measure_time.py \
    --train-manifest    data/librispeech/train-clean-100.manifest.jsonl \
    --train-transcripts data/librispeech/train-clean-100.transcripts.jsonl \
    --probe-steps 200

!cp runs/timing.json {DRIVE}/runs/

## 10. Chống mất session

Colab ngắt kết nối sau ~90 phút không tương tác. Chạy ô này ở tab riêng để giữ session.

In [ ]:
import IPython
from google.colab import output
IPython.display.display(IPython.display.Javascript('''
  function keepAlive(){
    document.querySelector("colab-toolbar-button#connect")?.click();
    console.log("keep-alive", new Date().toLocaleTimeString());
  }
  setInterval(keepAlive, 60000);
'''))
print("Keep-alive da bat.")

## Checklist cuối tuần 3

- [ ] `train-clean-100`: 28.539 utterance / ~100.6 giờ / 251 speaker
- [ ] `runs/anchor/anchor_result.json` có WER **20–30%**
- [ ] `runs/timing.json` có `sec_per_step` và tổng GPU-hour ước tính
- [ ] Cả hai file đã copy vào Drive
- [ ] Đã đọc 3 bài bổ sung: Huang 2016, Malhotra 2019, Nallasamy 2012

## Mẹo cho các tuần sau

1. **Đẩy code lên GitHub** — đỡ phải upload zip mỗi session.
2. **Luôn `cp` kết quả sang Drive ngay sau khi chạy xong** — `/content` mất khi ngắt session.
3. **Colab Pro** đáng cân nhắc từ tuần 9 trở đi (tuần 9 và 11 mỗi tuần cần ~25 GPU-hour).
4. **Kaggle** cho 2×T4 và ~30 giờ/tuần miễn phí — dùng xen kẽ với Colab để chia tải.